---
title: Keecas Quarto Example
toc: true
format:
    html:
        include-in-header:
            # this is needed for equation numbering and labeling in HTML output             
            - text: | 
                <script>
                MathJax = { tex: { tags: 'ams' } };
                </script>
            # this get rid of unnecessary vertical scrollbars (may need to be adjusted)
            - text: |
                <style>
                .math.display {
                  max-width: 100%;
                  padding-right: 3em;
                }
                </style>
    pdf: default
echo: false
keep-tex: false
---

# How to use keecas in a Quarto document

This example demonstrates key features of `keecas` for symbolic and units-aware calculations in a Quarto document.

In [178]:
# For quick start use starred import
# from keecas import *

# Or explicit import (this is about all you need)
from keecas import (
    check,                  # main function for verification
    config,                 # configuration options
    pc,                     # pipe commands
    show_eqn,               # main function to generate expression block
    symbols,                # to create symbols (from `sympy`)
    u,                      # unit registry (from pint)
    generate_unique_label,  # label generator
)

In [179]:
# Set the language ('en' is the default)
config.language.language = "es"

::: {.callout-note}

If the rendering engine is `KaTeX` (i.e. Jupyter notebook), the label command `\label{}` will result in a `ParseError`. The latex code will work when rendering by `quarto render`, but since you may want to see the result before rendering the document, an option to disable the `\label{}` command is provided.

When rendering with `quarto` you can have two config:

- if `qmd` files, then everything is fine
- if `ipynb` files, then pass the `--execute` flag to `quarto render` to rerender the notebook

:::


In [180]:
# | eval: false

# This option will not be set at rendering time (only for dev-mode)
config.display.katex = True         # Disable \label command for KaTeX
config.display.print_label = True   # Print labels for easy copy-paste

In [181]:
# Set a prefix for the latex equation (used as namespace)
config.latex.eq_prefix = r"eq-QUARTO_EXAMPLE-"
# config.latex.eq_suffix = ""  # default

# Initialize notebook-global dictionaries for persistence across cells
QUARTO_EXAMPLE = {                      # Main namespace dict for this notebook
    'parameters': (params := {}),       # Global parameters
    'expressions': (eqn := {}),         # Global expressions
    'values': (vals := {}),             # Global evaluated results
}

## Basic Symbolic Math with Units

Define symbols and calculate basic engineering quantities:

In [182]:
# Define symbols with LaTeX notation
F_d, A_load, sigma_Sd = symbols(r"F_{d}, A_{load}, \sigma_{Sd}")

# Cell-local parameters with units
_p = {
    F_d: 93 * u.kN,        # Applied force
    A_load: 50 * u.cm**2,  # Cross-sectional area
}
params.update(_p)  # Save to global params

# Cell-local symbolic expressions
_e = {
    sigma_Sd: "F_d / A_load" | pc.parse_expr,
}
eqn.update(_e)  # Save to global expressions

# Evaluate expressions
_v = {
    k: v | pc.subs(_e | _p) | pc.convert_to([u.MPa]) | pc.N
    for k, v in _e.items()
}

# Descriptions
_d = {
    F_d: "applied force",
    A_load: "cross-sectional area",
    sigma_Sd: "normal stress",
}

# Generate unique labels from keys and descriptions
_l = {
    k: generate_unique_label([k, v]) for k, v in _d.items()
}

show_eqn(
    [_p | _e, _v, _d],
    float_format="{:.2f}",
    label=_l,
    # debug=True,
)

# Labels will be printed for easy reference, only in dev-mode (not rendered by quarto)

F_{d}: eq-QUARTO_EXAMPLE-41g2kkvl
A_{load}: eq-QUARTO_EXAMPLE-3rad1giw
\sigma_{Sd}: eq-QUARTO_EXAMPLE-41gy5jxq


<IPython.core.display.Latex object>

## Pipe Commands Demonstration {#sec-pipe}

Using pipe operators for functional composition:

In [183]:
# Chain operations using pipe commands
x, y, d = symbols("x, y, d")

# Cell-local parameters
_p = {
    x: 3 * u.m,
    y: 4 * u.m,
}

# Symbolic expressions
_e = {
    d: "x^2 + y^2" | pc.parse_expr,
}

# Evaluated results
_v = {
    k: v | pc.subs(_e | _p) | pc.convert_to([u.m]) | pc.N
    for k, v in _e.items()
}

show_eqn([_p | _e, _v])

<IPython.core.display.Latex object>

## Different LaTeX Environments

### Cases Environment

Display related values grouped together:

In [184]:
# Material properties
E_steel, E_concrete = symbols(r"E_{steel}, E_{concrete}")

_p = {
    E_steel: 200 * u.GPa,
    E_concrete: 30 * u.GPa,
}

_d = {
    E_steel: "steel elastic modulus",
    E_concrete: "concrete elastic modulus",
}

show_eqn([_p, _d], environment="cases")

<IPython.core.display.Latex object>

### Equation Environment with Labels {#sec-beam-calc}

Calculate beam deflection with cross-references:

In [185]:
# Beam calculation
q, L, E, I, delta = symbols(r"q, L, E, I, \delta")

_p = {
    q: 5 * u.kN / u.m,
    L: 8 * u.m,
    E: 200 * u.GPa,
    I: 8360 * u.cm**4,
}
params.update(_p)  # Save to global params

# Deflection formula
_e = {
    delta: "5 * q * L^4 / (384 * E * I)" | pc.parse_expr,
}
eqn.update(_e)  # Save to global expressions

_v = {
    k: v | pc.subs(_e | _p) | pc.convert_to([u.mm]) | pc.N
    for k, v in _e.items()
}

_l = {
    k: generate_unique_label([k, v]) for k, v in _e.items()
}

show_eqn([_e, _v], label=_l, float_format="{:.2f}")

\delta: eq-QUARTO_EXAMPLE-2kp5fgk9


<IPython.core.display.Latex object>

::: {.callout-note}

labels emitted by `show_eqn` are latex labels, therefore they need to be referenced with `\eqref{}` or `\ref{}` latex commands.

@eq-QUARTO_EXAMPLE-delta will not work

:::

The deflection calculated in \ref{eq-QUARTO_EXAMPLE-delta} shows acceptable values for serviceability.

## Verification Function

Using the `check` function for design checks:

In [186]:
# Design checks
sigma_Sd, tau_Sd, sigma_Rd, tau_Rd = symbols(
    r"\sigma_{Sd}, \tau_{Sd}, \sigma_{Rd}, \tau_{Rd}",
)

_p = {
    sigma_Sd: 20 * u.MPa,
    tau_Sd: 15 * u.MPa,
    sigma_Rd: 250 * u.MPa,
    tau_Rd: 75 * u.MPa,
}

# Expressions to check
_expr = [
    sigma_Sd / sigma_Rd,
    tau_Sd / tau_Rd,
]

# Evaluate expressions
_v = {
    k: k | pc.subs(_e | _p) | pc.N for k in _expr
}

# Check if expressions are less than 1
_c = {
    k: check(v, 1.0) for k, v in _v.items()
}

# Specify float format only for the check values
_ff = {
    k: [None, "{:.3f}", None] for k in _c.keys()
}

show_eqn([_p | _v, _c], float_format=_ff)

<IPython.core.display.Latex object>

The type of comparison in the `check` function can be specified, as well as the value to be checked against:

In [187]:
from sympy import Eq, Ge, Gt, Le, Lt

a, b, c, d, f = symbols("a, b, c, d, f")

_expr = {
    a: (3, Le, 1),
    b: (4, Gt, 2),
    c: (5, Lt, 3),
    d: (6, Ge, 4),
    f: (7, Eq, 5),
}

_c = {
    k: check(lhs=v[0], test=v[1], rhs=v[2])
    for k, v in _expr.items()
}

show_eqn(
    [
        {k: v[0] for k, v in _expr.items()},
        _c,
    ],
)

<IPython.core.display.Latex object>

## Automatic Dependency Ordering

Using `pc.subs` will automatically order the substitutions pairs topologically, so you don't have to order them yourself (default ``sympy`` behavior). For example, let's define some mappings in any order:

In [188]:
any_order_params = {
    x: y**2, # it used `y` which is defined later
    y: a + 1, # `y` is a dependent value
    a: 3,
}

show_eqn(any_order_params, environment="cases")

<IPython.core.display.Latex object>

If we use the default behavior of `sympy`'s `subs` method (unsorted substitutions) we get:

In [189]:
# `pc.subs(any_order_params, sorted=False)` is equivalent to `sympy`'s subs method
_v = {
    lhs: rhs | pc.subs(any_order_params, sorted=False)
    for lhs, rhs in any_order_params.items()
}

_d = {
    x: "partially evaluated",
}

show_eqn(
    [_v, _d],
    environment="cases",
)

<IPython.core.display.Latex object>

While the default behavior of `pc.subs` is to sort the substitutions:

In [190]:
_v = {
    lhs: rhs | pc.subs(any_order_params)
    for lhs, rhs in any_order_params.items()
}

_d = {
    x: "fully evaluated",
}

show_eqn(
    [_v, _d],
    environment="cases",
)

<IPython.core.display.Latex object>

::: {.callout-note}

`pc.subs` automatically converts objects to `sympy` expressions before substitution. In the example above, $a$ maps to the Python `int` value `3`. When piping `int` through `pc.subs`, it works seamlessly. If you used `sympy`'s `subs()` method directly, you'd get an error since `int` objects don't have a `subs()` method. You'd need to write `S(rhs).subs(...)` to explicitly convert first.

:::

## Check Function Templates

The `check` function supports customizable templates for different visual styles:

In [191]:
# Default template behavior
result_default = [
    check(0.8, 1.0), # True
    check(1.2, 1.0), # False
]

# Using named template sets
result_boxed = [check(0.8, 1.0, template="boxed"), check(1.2, 1.0, template="boxed")]
result_minimal = [check(0.8, 1.0, template="minimal"), check(1.2, 1.0, template="minimal")]

# Demonstration of different template styles
template_demo = {
    "Default": result_default,
    "Boxed": result_boxed,
    "Minimal": result_minimal,
}

# template_demo is dict[Any, list], therefore it must converted to Dataframe before passing to show_eqn

from keecas import Dataframe

show_eqn(
    Dataframe(template_demo),
    col_wrap=[None, r":\quad", r"&\quad"],
    environment="align*",
)

<IPython.core.display.Latex object>

### Complex Expressions with Dependencies

Demonstrate how keecas handles expressions with dependencies:

In [192]:
# Complex expression with multiple substitutions
a, b, c, d = symbols(r"a, b, c, d")

_p = {
    a: 3,
    b: 4,
}
params.update(_p)

_e = {
    d: "sqrt(a^2 + b^2) / c" | pc.parse_expr,  # Uses c, which is defined below
    c: "a*b" | pc.parse_expr,                  # Defined after d, but keecas handles it
}
eqn.update(_e)

_v = {
    k: v | pc.subs(eqn | params) | pc.N
    for k, v in _e.items()
}

show_eqn([_p | _e, _v], float_format="{:.3f}")

<IPython.core.display.Latex object>

### Custom Templates

You can also use completely custom templates:

In [193]:
# Custom template examples (emoji may not render in latex)
custom_success = r"✅ ${symbol}{rhs}$ \textbf{{PASS}}"
custom_failure = r"❌ ${symbol}{rhs}$ \textbf{{FAIL}}"

# Test both success and failure cases
ratio_ok = 0.8      # Should pass
ratio_fail = 1.2    # Should fail

check_ok = check(
    ratio_ok,
    1.0,
    success_template=custom_success,
    failure_template=custom_failure,
)

check_fail = check(
    ratio_fail,
    1.0,
    success_template=custom_success,
    failure_template=custom_failure,
)

custom_demo = {
    "Pass": check_ok,
    "Fail": check_fail,
}

show_eqn(custom_demo, col_wrap=[None, ":", "&"])

<IPython.core.display.Latex object>

## Label Generation

Various methods for generating labels for equation cross-references:

In [194]:
# Setup for label examples
from keecas import generate_label

J, E, a, b, c_0 = symbols(r"J, E, a, b, c_{0}")

# Dictionary to be displayed by show_eqn
_p = {
    J: 123 * u.cm**4,
    E: 456 * u.MPa,
    a: "b + c_0 / 2" | pc.parse_expr,
}

### Manual Label Assignment

When manually assigning labels, they should be LaTeX-safe:

In [195]:
# When manually assigning labels, they should be LaTeX safe
_l = {
    J: "eq-moment-of-inertia",
    E: "eq-modulus-of-elasticity",
    a: "eq-an-expression",
}

# Pass _l to show_eqn
show_eqn(_p, label=_l)

J: eq-moment-of-inertia
E: eq-modulus-of-elasticity
a: eq-an-expression


<IPython.core.display.Latex object>

### Partial Automatic Generation

Generate labels with `eq-` prefix automatically:

#### Non-unique Labels

In [196]:
# Provide LaTeX-safe descriptions (no spaces)
_l = {
    J: "moment-of-inertia",
    E: "modulus-of-elasticity",
    a: "an-expression",
}

# Generate the labels with eq- prefix
_l = generate_label(_l)
print(_l)

# Pass _l to show_eqn
show_eqn(_p, label=_l)

{J: 'eq-QUARTO_EXAMPLE-moment-of-inertia', E: 'eq-QUARTO_EXAMPLE-modulus-of-elasticity', a: 'eq-QUARTO_EXAMPLE-an-expression'}
J: eq-QUARTO_EXAMPLE-moment-of-inertia
E: eq-QUARTO_EXAMPLE-modulus-of-elasticity
a: eq-QUARTO_EXAMPLE-an-expression


<IPython.core.display.Latex object>

#### Unique Labels

Generate hash-based unique labels:

In [197]:
# Descriptions with spaces (will be converted to LaTeX-safe format)
_d = {
    J: "moment of inertia",
    E: "modulus of elasticity",
    a: "an expression",
}

# Generate unique labels from descriptions
_l = generate_unique_label(_d)
print(_l)

# save to global dict for later retrieval
labels = _l

# Pass _l to show_eqn
show_eqn(_p, label=_l)

{J: 'eq-QUARTO_EXAMPLE-534vb10v', E: 'eq-QUARTO_EXAMPLE-1sipf20t', a: 'eq-QUARTO_EXAMPLE-30phqdq8'}
J: eq-QUARTO_EXAMPLE-534vb10v
E: eq-QUARTO_EXAMPLE-1sipf20t
a: eq-QUARTO_EXAMPLE-30phqdq8


<IPython.core.display.Latex object>

### Full Automatic Generation

A callable can be passed to `label` argument of `show_eqn`. For each key, all the key and values will be passed to the callable.

In [198]:
# Pass generate_unique_label directly to show_eqn
# Labels are generated from key + values from all the dict passed to show_eqn 

# this is equivalent 
_l = {
    k: generate_unique_label([k, _p[k], _d[k]]) for k in _p.keys()
}
print(_l)

# to this
show_eqn([_p, _d], label=generate_unique_label)

{J: 'eq-QUARTO_EXAMPLE-3lb6w5uv', E: 'eq-QUARTO_EXAMPLE-1mp6ez2n', a: 'eq-QUARTO_EXAMPLE-5kh184ai'}
J: eq-QUARTO_EXAMPLE-3lb6w5uv
E: eq-QUARTO_EXAMPLE-1mp6ez2n
a: eq-QUARTO_EXAMPLE-5kh184ai


<IPython.core.display.Latex object>

::: {.callout-note}

When using this method, the interpolated label will change if any of the values in the key-row of the Dataframe change. For a more stable approach, either pass an already interpolated dict of labels as string (recommended), or use a custom callable with a more resilient logic (e.g. use only the element of the list that are less likely to change).

Passing a dict of string as labels is recommended since labels are interpolated only once and they can be store and used for reference in the text, with an helper function

```{python}
from IPython.display import Markdown

# helper function for eqref
def eqref(label):
    return Markdown(r"\eqref{label}")
```

For example $J$ is defined at `{python} eqref(labels[J])`, while $E$ is defined at `{python} eqref(labels[E])`

:::

## Summary

This example demonstrated:

- Basic symbolic math with units
- Pipe command usage (see @sec-pipe)
- Different LaTeX environments (align, cases, equation)
- Label and cross-reference functionality
- Verification functions with `check()`
- Automatic dependency ordering
- Complex symbolic manipulations
- Template customization

All calculations in @sec-beam-calc show that keecas provides a powerful interface for engineering calculations in Quarto documents.

### Template Configuration

Templates can also be configured globally via TOML config files:

```toml
[check_templates.template_sets.minimal]
success = '${symbol}{rhs} \,\textcolor{{green}}{{\checkmark}}$'
failure = '${symbol}{rhs} \,\textcolor{{red}}{{\times}}$'
```

This allows you to set project-wide or user-wide styling for all check functions.

## Advanced: Custom Environment Definition

Test using a custom environment with raw LaTeX blocks:

In [200]:
a, b, c = symbols(r"a b c")

_p = {
    a: 5 * u.m,
    b: 12 * u.m,
    c: a + b,
}

_d = {
    a: "length a",
    b: "length b",
    c: "total length c",
}

# Custom environment definition with raw LaTeX blocks
custom_env = {
    'separator': ' & ',
    'line_separator': r' \\' + '\n',
    'supports_multiple_labels': True,
    'outer_environment': 'align',
    'inner_environment': None,
    'inner_prefix': '',
    'inner_suffix': '',
    'outer_prefix': '```{=latex}\n',
    'outer_suffix': '\n```',
    'label_position': 'outer',
}

show_eqn(
    [_p, _d],
    environment=custom_env,
    debug=True,
)

```{=latex}
\begin{align}
a  &  = 5{\,}\text{m}  &  \quad\text{length a}  \\[8pt]
b  &  = 12{\,}\text{m}  &  \quad\text{length b}  \\[8pt]
c  &  = a + b  &  \quad\text{total length c} 
\end{align}
```


<IPython.core.display.Latex object>